In [ ]:
import cv2
import numpy as np

# Load DNN face detector
face_model = 'C:\\Users\\sathi\\Downloads\\res10_300x300_ssd_iter_140000.caffemodel'
face_proto = 'C:\\Users\\sathi\\Downloads\\deploy.prototxt'
face_net = cv2.dnn.readNetFromCaffe(face_proto, face_model)

# Load age & gender models
age_proto = 'C:\\Users\\sathi\\Downloads\\age_deploy.prototxt'
age_model = 'C:\\Users\\sathi\\Downloads\\age_net.caffemodel'
gender_proto = 'C:\\Users\\sathi\\Downloads\\gender_deploy.prototxt'
gender_model = 'C:\\Users\\sathi\\Downloads\\gender_net.caffemodel'
age_net = cv2.dnn.readNetFromCaffe(age_proto, age_model)
gender_net = cv2.dnn.readNetFromCaffe(gender_proto, gender_model)

# Load Haar cascades
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
smile_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_smile.xml')

# Gender and Age labels
GENDER_LIST = ['Male', 'Female']
AGE_GROUPS = ['(0-15)', '(15-30)', '(30-40)']

# Face detector function
def get_face_box(net, frame, conf_threshold=0.7):
    h, w = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), [104, 117, 123], False, False)
    net.setInput(blob)
    detections = net.forward()
    boxes = []
    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > conf_threshold:
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            boxes.append(box.astype(int))
    return boxes

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = get_face_box(face_net, frame)

    for box in faces:
        x1, y1, x2, y2 = box
        # Safe clipping to frame boundaries
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(frame.shape[1], x2), min(frame.shape[0], y2)

        face_img = frame[y1:y2, x1:x2]
        if face_img.size == 0:
            continue

        # Prepare input for DNN
        blob = cv2.dnn.blobFromImage(face_img, 1.0, (227, 227),
                                     (78.426, 87.768, 114.895), swapRB=False)

        # Gender prediction
        gender_net.setInput(blob)
        gender_preds = gender_net.forward()
        gender = GENDER_LIST[gender_preds[0].argmax()]
        gender_conf = gender_preds[0].max()

        # Age prediction
        age_net.setInput(blob)
        age_preds = age_net.forward()[0]

        if len(age_preds) >= 6:
            age_group_values = {
                '(0-15)': float(age_preds[0]) + float(age_preds[1]) + float(age_preds[2]),
                '(15-30)': float(age_preds[3]) + float(age_preds[4]),
                '(30-40)': float(age_preds[5]),
            }
            double_chin_detected = False

            # Double Chin Detection (region below chin)
            chin_y1 = y2
            chin_y2 = min(y2 + (y2 - y1) // 3, frame.shape[0])
            chin_region = frame[chin_y1:chin_y2, x1:x2]

            if chin_region.size > 0:
                chin_gray = cv2.cvtColor(chin_region, cv2.COLOR_BGR2GRAY)
                chin_blur = cv2.GaussianBlur(chin_gray, (5, 5), 0)
                edges = cv2.Canny(chin_blur, 30, 100)

                contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                large_contours = [cnt for cnt in contours if cv2.contourArea(cnt) > 150]

                if len(large_contours) >= 2:
                    double_chin_detected = True
                    cv2.putText(frame, "Double Chin", (x1, chin_y2 + 15),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 100, 255), 1)
                    cv2.rectangle(frame, (x1, chin_y1), (x2, chin_y2), (200, 100, 255), 1)
                else:
                    cv2.putText(frame, "No Double Chin", (x1, chin_y2 + 15),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1)
                    cv2.rectangle(frame, (x1, chin_y1), (x2, chin_y2), (150, 150, 150), 1)

            # Influence age prediction
            if double_chin_detected:
                age_group_values['(30-40)'] += 0.15  # Heuristic

            age = max(age_group_values, key=age_group_values.get)
            age_conf = age_group_values[age]
        else:
            age = "Unknown"
            age_conf = 0.0

        # Draw gender & age
        label = f"{gender} ({gender_conf*100:.0f}%), Age: {age} ({age_conf*100:.0f}%)"
        color = (0, 255, 0) if gender == "Female" else (255, 0, 0)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # Eyes (Retina marker)
        face_gray = gray[y1:y2, x1:x2]
        eyes = eye_cascade.detectMultiScale(face_gray, 1.1, 10)
        for (ex, ey, ew, eh) in eyes[:2]:
            eye_center = (x1 + ex + ew // 2, y1 + ey + eh // 2)
            cv2.circle(frame, eye_center, 4, (0, 255, 255), -1)
            cv2.putText(frame, "Retina", (eye_center[0] + 5, eye_center[1]),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)

        # Mouth detection
        mouth_gray = face_gray[face_gray.shape[0] // 2:]
        mouths = smile_cascade.detectMultiScale(mouth_gray, 1.8, 20)
        for (mx, my, mw, mh) in mouths[:1]:
            cv2.rectangle(frame[y1:y2, x1:x2],
                          (mx, my + face_gray.shape[0] // 2),
                          (mx + mw, my + mh + face_gray.shape[0] // 2),
                          (255, 0, 255), 1)
            cv2.putText(frame[y1:y2, x1:x2], "Mouth",
                        (mx, my + face_gray.shape[0] // 2 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 255), 1)

    cv2.imshow("Age, Gender, Eyes, Mouth, Double Chin Detection", frame)
    if cv2.waitKey(1) & 0xFF == 27:  # ESC to exit
        break

cap.release()
cv2.destroyAllWindows()
